# 🚀 RoleNet V3 — Target: 80%+ Accuracy

## 🔑 Root Cause Analysis (V2 chỉ đạt 64.41%)

| Vấn đề | V2 | V3 Fix |
|--------|----|--------|
| **Dataset quá nhỏ** | Chỉ 2,374 ảnh (9% data) | **21,785 ảnh (100% data)** |
| **Architecture** | EfficientNet-B2 | **ConvNeXt-Tiny** (state-of-art) |
| **Scheduler** | CosineAnnealingLR | **OneCycleLR** (warmup + aggressive decay) |
| **Augmentation** | Basic MixUp/CutMix | **RandAugment + TrivialAugment** |
| **Class balance** | WeightedLoss | **WeightedRandomSampler + WeightedLoss** |

## 📌 Kiến trúc ConvNeXt-Tiny
- ImageNet-1k top-1: **82.1%** (vs EfficientNet-B2: 80.4%)
- Params: 28M (vs B2: 9.1M) — GPU T4 vẫn handle OK
- Thiết kế hiện đại: depthwise conv + LayerNorm thay BN

---
⚡ **Yêu cầu**: Runtime → T4 GPU | Upload `rolenet_upload_v3.zip` (splits_v3 + augmented) lên Google Drive

## 📌 Bước 1: Mount Drive & Giải Nén Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

# ── Chọn 1 trong 2 cách truyền data ──────────────────────────────────────
# Cách A: dùng file mới rolenet_upload_v3.zip (splits_v3 + augmented)
ZIP_V3   = '/content/drive/MyDrive/rolenet_upload_v3.zip'
# Cách B: dùng file cũ rolenet_upload.zip (augmented đầy đủ)
ZIP_V1   = '/content/drive/MyDrive/rolenet_upload.zip'

EXTRACT_DIR = '/content/rolenet_dataset'

if not os.path.exists(EXTRACT_DIR):
    # Ưu tiên V3 zip trước
    if os.path.exists(ZIP_V3):
        print(f'📦 Giải nén {ZIP_V3}...')
        os.system(f'unzip -q "{ZIP_V3}" -d /content/')
    elif os.path.exists(ZIP_V1):
        print(f'📦 Giải nén {ZIP_V1}...')
        os.system(f'unzip -q "{ZIP_V1}" -d /content/')
    else:
        raise FileNotFoundError('Không tìm thấy zip file trên Drive!')
    print('✅ Giải nén xong!')
else:
    print('✅ Dataset đã tồn tại.')

# Đếm ảnh
aug_dir = Path(EXTRACT_DIR) / 'augmented'
total = sum(len(list(d.glob('*.jpg'))) + len(list(d.glob('*.png')))
            for d in aug_dir.iterdir() if d.is_dir())
print(f'📊 Tổng ảnh augmented: {total:,}')
os.system(f'ls {EXTRACT_DIR}')

## 📌 Bước 2: Khởi Tạo Splits V3 (21,785 ảnh train)

In [ ]:
# Tạo splits_v3 nếu chưa có (dùng tất cả 27k ảnh)
import pandas as pd
import numpy as np
import random
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split

splits_v3 = Path('/content/rolenet_dataset/splits_v3')

CLASS_NAMES = [
    'shipper','doctor','police','military','security','student',
    'chef','janitor','construction','nurse','postman','technician',
    'worker','civil_guard','normal','unknown'
]

if splits_v3.exists() and (splits_v3 / 'train.csv').exists():
    print('✅ splits_v3 đã tồn tại.')
else:
    print('🔨 Tạo splits_v3 từ toàn bộ augmented data...')
    splits_v3.mkdir(exist_ok=True)
    random.seed(42); np.random.seed(42)

    aug_dir = Path('/content/rolenet_dataset/augmented')
    all_paths, all_labels = [], []

    for cls_name in CLASS_NAMES:
        cls_dir = aug_dir / cls_name
        if not cls_dir.exists():
            continue
        for img in list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')):
            all_paths.append(f'{cls_name}/{img.name}')
            all_labels.append(cls_name)

    print(f'  Total: {len(all_paths):,} images')

    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        all_paths, all_labels, test_size=0.2, stratify=all_labels, random_state=42)
    X_val, X_te, y_val, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

    pd.DataFrame({'path': X_tr,  'label': y_tr }).to_csv(splits_v3 / 'train.csv', index=False)
    pd.DataFrame({'path': X_val, 'label': y_val}).to_csv(splits_v3 / 'val.csv',   index=False)
    pd.DataFrame({'path': X_te,  'label': y_te }).to_csv(splits_v3 / 'test.csv',  index=False)
    print('✅ splits_v3 created!')

# Hiển thị stats
for split_name in ['train', 'val', 'test']:
    df = pd.read_csv(splits_v3 / f'{split_name}.csv')
    print(f'  {split_name:5s}: {len(df):6,} samples | {df["label"].nunique()} classes')

## 📌 Bước 3: Cài Thư Viện

In [ ]:
!pip install -q timm torchmetrics scikit-learn --upgrade
import torch, timm
print(f'PyTorch : {torch.__version__}')
print(f'timm    : {timm.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ No GPU"}')
print(f'CUDA    : {torch.version.cuda}')

# Kiểm tra memory
if torch.cuda.is_available():
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU Mem : {mem:.1f} GB')

## 📌 Bước 4: Config V3

In [ ]:
import torch, timm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from collections import Counter
import json, time, copy, random

# ============================================================
# CONFIG V3 — Key changes vs V2:
#   1. data_root: augmented (all 27k images)
#   2. splits_dir: splits_v3 (80/10/10)
#   3. model_name: convnext_tiny (higher capacity)
#   4. epochs: 5+15+30 = 50 total (vs V2: 5+10+20=35)
#   5. batch_size: 64 (T4 can handle)
#   6. use_sampler: WeightedRandomSampler for balance
# ============================================================
CFG = {
    # Dataset
    'data_root'    : '/content/rolenet_dataset/augmented',
    'splits_dir'   : '/content/rolenet_dataset/splits_v3',
    'img_size'     : (256, 128),

    # Model — ConvNeXt-Tiny thay EfficientNet-B2
    'model_name'   : 'convnext_tiny',
    'num_classes'  : 16,
    'pretrained'   : True,
    'drop_rate'    : 0.3,

    # Training phases
    'phase1_epochs': 5,    # Head only
    'phase2_epochs': 15,   # Top 2 stages
    'phase3_epochs': 30,   # Full fine-tune

    'batch_size'   : 64,
    'lr_head'      : 1e-3,
    'lr_backbone'  : 5e-5,
    'weight_decay' : 1e-4,
    'label_smooth' : 0.1,

    # Augmentation
    'mixup_alpha'  : 0.4,
    'cutmix_alpha' : 1.0,
    'mixup_prob'   : 0.5,

    # Output
    'save_dir'     : '/content/drive/MyDrive/rolenet_v3_checkpoints',
    'device'       : 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers'  : 4,
    'seed'         : 42,
}

CLASS_NAMES = [
    'shipper', 'doctor', 'police', 'military', 'security', 'student',
    'chef', 'janitor', 'construction', 'nurse', 'postman', 'technician',
    'worker', 'civil_guard', 'normal', 'unknown'
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
random.seed(CFG['seed'])

Path(CFG['save_dir']).mkdir(parents=True, exist_ok=True)

total_epochs = CFG['phase1_epochs'] + CFG['phase2_epochs'] + CFG['phase3_epochs']
print('✅ Config V3 loaded!')
print(f'   Model   : {CFG["model_name"]}')
print(f'   Device  : {CFG["device"]}')
print(f'   Epochs  : {total_epochs} (P1={CFG["phase1_epochs"]}, P2={CFG["phase2_epochs"]}, P3={CFG["phase3_epochs"]})')
print(f'   Classes : {CFG["num_classes"]}')
print(f'   Batch   : {CFG["batch_size"]}')

## 📌 Bước 5: Dataset & DataLoader (Full 21k train)

In [ ]:
class RoleNetDataset(Dataset):
    def __init__(self, root_dir, split_csv, transform=None):
        self.root      = Path(root_dir)
        self.transform = transform
        self.samples   = []
        df = pd.read_csv(split_csv)
        for _, row in df.iterrows():
            # Handle both backslash (Windows) and forward slash
            rel_path = str(row['path']).replace('\\', '/')
            p = self.root / rel_path
            l = CLASS_TO_IDX.get(row['label'], -1)
            if p.exists() and l >= 0:
                self.samples.append((str(p), l))
        print(f'  Loaded {len(self.samples):,} from {Path(split_csv).name}')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── V3 Augmentation: thêm TrivialAugmentWide ──────────────────
train_transform = transforms.Compose([
    transforms.Resize(CFG['img_size']),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.TrivialAugmentWide(),          # Mạnh hơn RandAugment
    transforms.RandomApply([
        transforms.RandomPerspective(distortion_scale=0.3)
    ], p=0.3),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.25)),
])

val_transform = transforms.Compose([
    transforms.Resize(CFG['img_size']),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('📂 Loading datasets...')
splits = CFG['splits_dir']
train_ds = RoleNetDataset(CFG['data_root'], f'{splits}/train.csv', train_transform)
val_ds   = RoleNetDataset(CFG['data_root'], f'{splits}/val.csv',   val_transform)
test_ds  = RoleNetDataset(CFG['data_root'], f'{splits}/test.csv',  val_transform)

# ── WeightedRandomSampler để cân bằng class ───────────────────
labels_list = [s[1] for s in train_ds.samples]
counts = Counter(labels_list)
total_n = len(labels_list)
# Weight mỗi sample ~ 1/(count của class đó)
sample_weights = [1.0 / counts[lbl] for lbl in labels_list]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=total_n,
    replacement=True
)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], sampler=sampler,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)

print(f'\n✅ Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')
print(f'   WeightedRandomSampler: active (balanced batches)')

# Class distribution
print('\n📊 Class distribution (train):')
for i, name in enumerate(CLASS_NAMES):
    bar = '█' * (counts[i] // 100)
    print(f'  {name:15s} {counts[i]:5d}  {bar}')

# Class weights cho loss
class_wts = torch.tensor([total_n / (16.0 * counts[i]) for i in range(16)],
                          dtype=torch.float32).to(CFG['device'])
print(f'\n⚖️  Class weights: min={class_wts.min():.2f}, max={class_wts.max():.2f}')

## 📌 Bước 6: Model — ConvNeXt-Tiny

In [ ]:
def build_rolenet_v3(num_classes=16, pretrained=True, drop_rate=0.3):
    """
    RoleNet V3: ConvNeXt-Tiny backbone + custom classification head.
    - ConvNeXt-Tiny: 28M params, ImageNet top-1=82.1%
    - Head: AdaptiveAvgPool → LayerNorm → Dropout → Linear(768→256) → GELU → Linear(256→16)
    """
    backbone = timm.create_model(
        'convnext_tiny',
        pretrained=pretrained,
        num_classes=0,       # Remove head
        drop_rate=drop_rate,
    )
    in_features = backbone.num_features  # 768 for ConvNeXt-Tiny

    # Classification head với LayerNorm (phù hợp với ConvNeXt)
    head = nn.Sequential(
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.LayerNorm(in_features),
        nn.Dropout(p=drop_rate),
        nn.Linear(in_features, 256),
        nn.GELU(),
        nn.Dropout(p=drop_rate * 0.5),
        nn.Linear(256, num_classes),
    )

    class RoleNetV3(nn.Module):
        def __init__(self, backbone, classifier):
            super().__init__()
            self.backbone   = backbone
            self.classifier = classifier

        def forward(self, x):
            feat = self.backbone.forward_features(x)
            return self.classifier(feat)

        def freeze_backbone(self):
            for p in self.backbone.parameters():
                p.requires_grad = False

        def unfreeze_top_stages(self, n_stages=2):
            """Unfreeze n stages cuối của ConvNeXt (4 stages total)."""
            stages = list(self.backbone.stages)
            for stage in stages[-n_stages:]:
                for p in stage.parameters():
                    p.requires_grad = True
            # Norm head luôn unfreeze
            if hasattr(self.backbone, 'norm_pre'):
                for p in self.backbone.norm_pre.parameters():
                    p.requires_grad = True
            if hasattr(self.backbone, 'head'):
                for p in self.backbone.head.parameters():
                    p.requires_grad = True

        def unfreeze_all(self):
            for p in self.parameters():
                p.requires_grad = True

    net = RoleNetV3(backbone, head)
    return net


model = build_rolenet_v3(
    num_classes=CFG['num_classes'],
    pretrained=CFG['pretrained'],
    drop_rate=CFG['drop_rate'],
).to(CFG['device'])

total_p    = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model: ConvNeXt-Tiny + Custom Head')
print(f'   Total params    : {total_p:,} (~{total_p*4/1024/1024:.1f} MB)')
print(f'   In features     : {model.backbone.num_features}')

# Test forward
with torch.no_grad():
    dummy = torch.randn(2, 3, 256, 128).to(CFG['device'])
    out   = model(dummy)
    print(f'   Output shape    : {out.shape} ✅')

## 📌 Bước 7: MixUp & CutMix

In [ ]:
def mixup_data(x, y, alpha=0.4, device='cuda'):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    bs  = x.size(0)
    idx = torch.randperm(bs).to(device)
    x_mix  = lam * x + (1 - lam) * x[idx]
    return x_mix, y, y[idx], lam


def cutmix_data(x, y, alpha=1.0, device='cuda'):
    lam = np.random.beta(alpha, alpha)
    bs  = x.size(0)
    idx = torch.randperm(bs).to(device)
    _, _, H, W = x.shape
    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, y1 = max(cx - cut_w // 2, 0), max(cy - cut_h // 2, 0)
    x2, y2 = min(cx + cut_w // 2, W), min(cy + cut_h // 2, H)
    x_cut = x.clone()
    x_cut[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    return x_cut, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print('✅ MixUp & CutMix ready.')

## 📌 Bước 8: Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scheduler, device, use_mixup=True):
    model.train()
    total_loss = correct = n = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        use_mix = use_mixup and random.random() < CFG['mixup_prob']
        if use_mix:
            if random.random() < 0.5:
                imgs, y_a, y_b, lam = mixup_data(imgs, labels, CFG['mixup_alpha'], device)
            else:
                imgs, y_a, y_b, lam = cutmix_data(imgs, labels, CFG['cutmix_alpha'], device)

        optimizer.zero_grad()
        outputs = model(imgs)

        if use_mix:
            loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
        else:
            loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item() * imgs.size(0)
        preds       = outputs.argmax(1)
        correct    += (preds == labels).sum().item()
        n          += imgs.size(0)

    return total_loss / n, correct / n


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = correct = n = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        n          += imgs.size(0)
    return total_loss / n, correct / n


def make_optimizer(model, phase):
    if phase == 1:
        params = filter(lambda p: p.requires_grad, model.parameters())
        return optim.AdamW(params, lr=CFG['lr_head'], weight_decay=CFG['weight_decay'])
    elif phase == 2:
        return optim.AdamW([
            {'params': model.classifier.parameters(),                     'lr': CFG['lr_head']},
            {'params': filter(lambda p: p.requires_grad,
                              model.backbone.parameters()),               'lr': CFG['lr_backbone'] * 5},
        ], weight_decay=CFG['weight_decay'])
    else:  # phase 3
        return optim.AdamW([
            {'params': model.classifier.parameters(),  'lr': CFG['lr_head'] * 0.5},
            {'params': model.backbone.parameters(),    'lr': CFG['lr_backbone']},
        ], weight_decay=CFG['weight_decay'])


print('✅ Training functions ready.')

## 📌 Bước 9: Training — 3 Phase Progressive Unfreezing (50 epochs)

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_wts,
    label_smoothing=CFG['label_smooth']
)

history  = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'phase': []}
best_acc = 0.0
best_weights = None
save_path = Path(CFG['save_dir'])


def run_phase(phase_num, n_epochs, use_mixup=False):
    global best_acc, best_weights

    optimizer = make_optimizer(model, phase_num)
    # OneCycleLR: warmup + cosine decay — mạnh hơn CosineAnnealingLR
    max_lr = optimizer.param_groups[0]['lr']
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[pg['lr'] for pg in optimizer.param_groups],
        steps_per_epoch=len(train_loader),
        epochs=n_epochs,
        pct_start=0.1,      # 10% warmup
        anneal_strategy='cos',
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"="*70}')
    print(f'  PHASE {phase_num} | {n_epochs} epochs | MixUp={use_mixup} | Trainable: {trainable:,}')
    print(f'{"="*70}')

    for ep in range(1, n_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer,
                                       scheduler, CFG['device'], use_mixup)
        vl_loss, vl_acc = eval_epoch(model, val_loader, criterion, CFG['device'])

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)
        history['phase'].append(phase_num)

        is_best = vl_acc > best_acc
        if is_best:
            best_acc     = vl_acc
            best_weights = copy.deepcopy(model.state_dict())
            torch.save({
                'epoch'      : len(history['val_acc']),
                'model_state': model.state_dict(),
                'val_acc'    : vl_acc,
                'classes'    : CLASS_NAMES,
                'phase'      : phase_num,
                'arch'       : CFG['model_name'],
            }, save_path / 'rolenet_v3_best.pt')

        lr_now  = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0
        marker  = ' ⭐ BEST' if is_best else ''
        print(
            f'  P{phase_num} Ep {ep:2d}/{n_epochs} | '
            f'Loss {tr_loss:.4f}/{vl_loss:.4f} | '
            f'Acc {tr_acc*100:5.1f}%/{vl_acc*100:5.1f}% | '
            f'LR {lr_now:.1e} | {elapsed:.0f}s{marker}'
        )


t_start = time.time()

# ─── PHASE 1: Head only ─────────────────────────────────────────
print('🔒 PHASE 1: Freeze backbone → train head only...')
model.freeze_backbone()
run_phase(1, CFG['phase1_epochs'], use_mixup=False)

# ─── PHASE 2: Unfreeze top 2 stages ───────────────────────────────────
print('\n🔓 PHASE 2: Unfreeze top 2 stages...')
model.unfreeze_top_stages(n_stages=2)
run_phase(2, CFG['phase2_epochs'], use_mixup=True)

# ─── PHASE 3: Full fine-tune ────────────────────────────────────────────
print('\n🔓 PHASE 3: Unfreeze toàn bộ...')
model.unfreeze_all()
run_phase(3, CFG['phase3_epochs'], use_mixup=True)

total_time = (time.time() - t_start) / 60
print(f'\n✅ Training hoàn tất! {total_time:.1f} phút')
print(f'   🏆 Best Val Accuracy: {best_acc*100:.2f}%')
print(f'   📁 Saved → {save_path}/rolenet_v3_best.pt')

## 📌 Bước 10: Learning Curve

In [ ]:
epochs_ran = len(history['train_acc'])
x = range(1, epochs_ran + 1)

p1_end = CFG['phase1_epochs']
p2_end = p1_end + CFG['phase2_epochs']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(x, history['train_loss'], 'b-',  label='Train Loss', lw=2)
ax1.plot(x, history['val_loss'],   'r-',  label='Val Loss',   lw=2)
ax1.axvline(p1_end, color='orange', ls='--', alpha=0.7, label='Phase 1→2')
ax1.axvline(p2_end, color='green',  ls='--', alpha=0.7, label='Phase 2→3')
ax1.set_title('Loss Curve', fontsize=13)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(x, [a*100 for a in history['train_acc']], 'b-', label='Train Acc', lw=2)
ax2.plot(x, [a*100 for a in history['val_acc']],   'r-', label='Val Acc',   lw=2)
ax2.axhline(best_acc*100, color='gold',  ls='--', lw=2, label=f'Best {best_acc*100:.1f}%')
ax2.axhline(64.41,        color='orange', ls=':',  lw=1, label='V2 baseline 64%')
ax2.axhline(80.0,         color='lime',  ls=':',  lw=1, label='Target 80%')
ax2.axvline(p1_end, color='orange', ls='--', alpha=0.7)
ax2.axvline(p2_end, color='green',  ls='--', alpha=0.7)
ax2.set_title('Accuracy Curve', fontsize=13)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_ylim(0, 105)

plt.suptitle(
    f'RoleNet V3 (ConvNeXt-Tiny) — Best: {best_acc*100:.1f}%  '
    f'(V2: 64.4% | V1: 59.0%)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('/content/training_v3.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Chart saved → /content/training_v3.png')

## 📌 Bước 11: Đánh Giá Test Set + Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.load_state_dict(best_weights)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs.to(CFG['device'])).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print('📊 Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3))

cm   = confusion_matrix(all_labels, all_preds)
cm_p = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_p, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, cbar_kws={'label': '%'})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion Matrix — RoleNet V3 (Best Acc: {best_acc*100:.1f}%)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/confusion_v3.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved → /content/confusion_v3.png')

## 📌 Bước 12: Test-Time Augmentation (TTA) Evaluation

In [ ]:
@torch.no_grad()
def predict_tta(model, imgs_tensor, device):
    """TTA: 6 augmentations → average probabilities."""
    model.eval()
    imgs = imgs_tensor.to(device)
    probs = torch.softmax(model(imgs), dim=1)
    # Horizontal flip
    probs += torch.softmax(model(imgs.flip(-1)), dim=1)
    # Brightness variations
    for factor in [0.85, 1.15]:
        probs += torch.softmax(model((imgs * factor).clamp(0, 1)), dim=1)
    # Center crop
    _, _, H, W = imgs.shape
    cr = int(0.08 * min(H, W))
    cropped = imgs[:, :, cr:H-cr, cr:W-cr]
    resized  = torch.nn.functional.interpolate(cropped, size=(H, W), mode='bilinear', align_corners=False)
    probs += torch.softmax(model(resized), dim=1)
    # Vertical flip (useful for person crops)
    probs += torch.softmax(model(imgs.flip(-2)), dim=1)
    return probs / 6.0


print('🔮 Đánh giá với TTA...')
model.load_state_dict(best_weights)
all_preds_tta, all_lbls_tta = [], []

for imgs, labels in test_loader:
    probs = predict_tta(model, imgs, CFG['device'])
    all_preds_tta.extend(probs.argmax(1).cpu().numpy())
    all_lbls_tta.extend(labels.numpy())

from sklearn.metrics import accuracy_score
acc_no_tta = accuracy_score(all_labels, all_preds)
acc_tta    = accuracy_score(all_lbls_tta, all_preds_tta)

print(f'\n📊 Kết quả test set:')
print(f'   Accuracy (no TTA) : {acc_no_tta*100:.2f}%')
print(f'   Accuracy (TTA)    : {acc_tta*100:.2f}%')
print(f'   TTA boost         : +{(acc_tta - acc_no_tta)*100:.2f}%')
print(f'   V2 baseline       : 64.41%')
print(f'   V1 baseline       : 59.05%')
print(f'   Tổng cải thiện    : +{(acc_tta - 0.5905)*100:.2f}% (so với V1)')

## 📌 Bước 13: Export Model

In [ ]:
from sklearn.metrics import accuracy_score
model.load_state_dict(best_weights)
model.eval().cpu()

export_dir  = Path(CFG['save_dir'])
dummy_input = torch.randn(1, 3, 256, 128)

# ── TorchScript ──────────────────────────────────────────────────
try:
    scripted = torch.jit.trace(model, dummy_input)
    ts_path  = export_dir / 'rolenet_v3_scripted.pt'
    scripted.save(str(ts_path))
    sz = ts_path.stat().st_size / 1024 / 1024
    print(f'✅ TorchScript → {ts_path} ({sz:.1f} MB)')
except Exception as e:
    print(f'⚠️  TorchScript failed: {e}')

# ── Metadata JSON ───────────────────────────────────────────────
meta = {
    'model'         : 'ConvNeXt-Tiny',
    'version'       : 'v3',
    'num_classes'   : CFG['num_classes'],
    'classes'       : CLASS_NAMES,
    'class_to_idx'  : CLASS_TO_IDX,
    'input_size'    : list(CFG['img_size']),
    'best_val_acc'  : round(best_acc * 100, 2),
    'v2_acc'        : 64.41,
    'v1_acc'        : 59.05,
    'improvement_vs_v2': round((best_acc - 0.6441) * 100, 2),
    'imagenet_mean' : [0.485, 0.456, 0.406],
    'imagenet_std'  : [0.229, 0.224, 0.225],
    'label_map'     : {str(i): c for i, c in enumerate(CLASS_NAMES)},
    'tta_enabled'   : True,
    'train_samples' : 21785,
    'arch_details'  : {
        'backbone'   : 'convnext_tiny',
        'in_features': 768,
        'head'       : 'LayerNorm→Dropout→Linear(768→256)→GELU→Linear(256→16)',
    }
}

meta_path = export_dir / 'rolenet_v3_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(f'✅ Metadata → {meta_path}')

print(f'\n🎉 Export hoàn tất!')
print(f'   rolenet_v3_best.pt      : checkpoint đầy đủ')
print(f'   rolenet_v3_scripted.pt  : TorchScript (production)')
print(f'   rolenet_v3_metadata.json: metadata & class map')
print(f'\n   🏆 Best Val Accuracy: {best_acc*100:.2f}% (target: ≥80%)')

## 📌 Bước 14: Download Files

In [ ]:
import os, shutil
from pathlib import Path

# Copy 3 file quan trọng ra /content để download
save_p = Path(CFG['save_dir'])
download_dir = Path('/content/rolenet_v3_download')
download_dir.mkdir(exist_ok=True)

for fname in ['rolenet_v3_best.pt', 'rolenet_v3_scripted.pt', 'rolenet_v3_metadata.json']:
    src = save_p / fname
    dst = download_dir / fname
    if src.exists():
        shutil.copy(src, dst)
        sz = dst.stat().st_size / 1024 / 1024
        print(f'  ✅ {fname} ({sz:.1f} MB)')
    else:
        print(f'  ❌ {fname} not found!')

print('\n📥 Download từ Colab:')
from google.colab import files
for fname in ['rolenet_v3_best.pt', 'rolenet_v3_scripted.pt', 'rolenet_v3_metadata.json']:
    fpath = download_dir / fname
    if fpath.exists():
        files.download(str(fpath))

print('\n📋 Sau khi download, copy vào local:')
print('   copy rolenet_v3_best.pt       c:\\code\\camera-ai\\models\\')
print('   copy rolenet_v3_metadata.json c:\\code\\camera-ai\\models\\')
print('   copy rolenet_v3_scripted.pt   c:\\code\\camera-ai\\models\\')
print('\n   Rồi update role_classifier.py để nhận V3.')